# PathSnap — Model 3: Random Forest

Same task, data, folds and metrics as Models 1 and 2 (logistic regression, HistGradientBoosting), so the results compare directly.
Each row is one (GPS observation, candidate road arc) pair with 24 features; the label is 1 if the arc is the observation's reference arc.

**Why a random forest here:** it is the *ensemble* family — many decorrelated trees, each trained on a bootstrap sample with random feature subsets, averaged. Boosting fits trees one after another to correct errors; a forest builds them independently. Comparing the two shows whether the gain over the linear model comes from tree interactions in general or from boosting specifically, and how stable each is across folds.

**Probability calibration:** averaged tree votes are usually under-confident near 0 and 1. The HMM needs well-calibrated probabilities, so a Platt (sigmoid) calibration is fitted on the **out-of-fold** cross-validation predictions — trips the forest never saw — and applied to student_test. Calibration is monotonic, so it never changes which candidate ranks first (top-1); it only improves log loss.

**Runtime on Colab (2 CPU cores):** about 1–2 minutes per fit, 6 fits in total, roughly 10–15 minutes for the whole notebook.

**Before running on Colab:** `pathsnap_features.zip` in `MyDrive/PathSnap/` (same as Models 1 and 2). The optional map cell at the end also needs `Student folder.zip` there.

In [ ]:
# ---- Paths: Colab (Drive zip) or local run --------------------------------------------
import os, sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP = Path("/content/drive/MyDrive/PathSnap/pathsnap_features.zip")
    DATA = Path("/content/pathsnap_features")
    RESULTS = Path("/content/drive/MyDrive/PathSnap/results")
    if not (DATA / "data_features").exists():
        with zipfile.ZipFile(ZIP) as z:
            z.extractall(DATA)
else:
    DATA = Path(os.environ.get("PATHSNAP_DATA", "../.."))
    RESULTS = Path(os.environ.get("PATHSNAP_RESULTS", "results"))

OUT = RESULTS / "random_forest"
FEAT_DIR = DATA / "data_features"
FLAGS = DATA / "trip_flags.csv" if (DATA / "trip_flags.csv").exists() else DATA / "data_clean" / "trip_flags.csv"
OUT.mkdir(parents=True, exist_ok=True)
print("features:", FEAT_DIR, "| flags:", FLAGS, "| output:", OUT)

In [ ]:
import json, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, log_loss, ConfusionMatrixDisplay)

SEED = 42
np.random.seed(SEED)

FEATURES = json.loads((FEAT_DIR / "feature_params.json").read_text())["FEATURES"]

CONFIG = {
    "model": "RandomForestClassifier",
    "family": "Ensemble (bagging)",
    "scaler": "none (trees are scale-invariant)",
    "n_estimators": 200,
    "max_features": "sqrt",     # ~5 of 24 features tried per split: decorrelates the trees
    "min_samples_leaf": 20,     # stops trees memorising single noisy GPS points; keeps the model ~65 MB
    "max_depth": None,
    "bootstrap": True,
    "max_samples": 0.25,        # each tree sees a 25% bootstrap sample: 4x faster, more diverse trees
    "class_weight": None,
    "calibration": "Platt sigmoid on out-of-fold CV predictions",
    "cv": "GroupKFold(n_splits=5) on trip_id",
    "seed": SEED,
    "n_jobs": -1,
    "n_features": len(FEATURES),
}

def make_model(cfg=CONFIG):
    return RandomForestClassifier(
        n_estimators=cfg["n_estimators"], max_features=cfg["max_features"],
        min_samples_leaf=cfg["min_samples_leaf"], max_depth=cfg["max_depth"],
        bootstrap=cfg["bootstrap"], max_samples=cfg["max_samples"],
        class_weight=cfg["class_weight"], n_jobs=cfg["n_jobs"], random_state=cfg["seed"])

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)

print(f"sklearn {sklearn.__version__} | {len(FEATURES)} features")

## 1. Load data
Identical to Models 1 and 2: usable train trips only (`use_for_tuning`), rows with `label = -1` dropped, per-observation reference info kept so observations without a correct candidate count as wrong.

In [ ]:
flags = pd.read_csv(FLAGS, dtype={"trip_id": str})

def trips_of(part):
    t = flags[flags.partition == part]
    if part == "train":
        t = t[t.use_for_tuning.astype(bool)]
    return sorted(t.trip_id)

def load_partition(part):
    rows, pts = [], []
    for tid in trips_of(part):
        f = pd.read_csv(FEAT_DIR / part / f"{tid}.csv.gz", dtype={"trip_id": str})
        y = pd.read_csv(FEAT_DIR / "labels" / part / f"{tid}.csv.gz")
        assert len(f) == len(y) and (f.idx.values == y.idx.values).all() and (f.arc_id.values == y.arc_id.values).all()
        f["label"] = y.label.values
        rows.append(f[f.label >= 0])
        p = pd.read_csv(FEAT_DIR / "labels" / part / f"{tid}_points.csv.gz")
        p.insert(0, "trip_id", tid)
        pts.append(p[p.labelled])
    return pd.concat(rows, ignore_index=True), pd.concat(pts, ignore_index=True)

t0 = time.time()
train, train_pts = load_partition("train")
test, test_pts = load_partition("student_test")
print(f"loaded in {time.time() - t0:.1f}s")
for name, d, p in [("train", train, train_pts), ("student_test", test, test_pts)]:
    print(f"{name:13} trips {d.trip_id.nunique():3} | rows {len(d):>9,} | positive rate {d.label.mean():.3f} "
          f"| labelled observations {len(p):>7,} | candidate miss rate {1 - p.ref_in_candidates.mean():.4%}")

## 2. Metrics
Same definitions as Models 1 and 2. **Primary metric: observation-level top-1 accuracy** (no correct candidate = wrong). Row-level metrics use a 0.5 threshold.

In [ ]:
def row_metrics(y, p, thr=0.5):
    yhat = (p >= thr).astype(int)
    return {
        "accuracy": accuracy_score(y, yhat),
        "precision": precision_score(y, yhat, zero_division=0),
        "recall": recall_score(y, yhat, zero_division=0),
        "f1": f1_score(y, yhat, zero_division=0),
        "log_loss": log_loss(y, np.clip(p, 1e-7, 1 - 1e-7), labels=[0, 1]),
    }

def top1(df, score, pts):
    d = df[["trip_id", "idx", "label"]].assign(s=score)
    best = d.loc[d.groupby(["trip_id", "idx"]).s.idxmax()]
    hits = int(best.label.sum())
    sel = pts[pts.trip_id.isin(df.trip_id.unique())]
    return {"top1": hits / len(sel), "top1_given_candidate": hits / max(int(sel.ref_in_candidates.sum()), 1), "n_obs": len(sel)}

def evaluate(df, p, pts):
    m = row_metrics(df.label.values, p)
    m.update(top1(df, p, pts))
    m["top1_nearest_arc"] = top1(df, -df.dist_rank.values.astype(float), pts)["top1"]
    return m

## 3. Cross-validation — GroupKFold by trip (train only)
Same 5 folds as Models 1 and 2 (GroupKFold is deterministic for the same data), so fold-by-fold results are directly comparable. Each fold's validation predictions are also kept as **out-of-fold** predictions for the calibration step. Metrics below are for the raw (uncalibrated) forest.

In [ ]:
gkf = GroupKFold(n_splits=5)
cv_rows = []
oof = np.zeros(len(train))
for fold, (tr, va) in enumerate(gkf.split(train, train.label, groups=train.trip_id)):
    a, b = train.iloc[tr], train.iloc[va]
    model = make_model()
    t0 = time.time(); model.fit(a[FEATURES], a.label); fit_s = time.time() - t0
    t0 = time.time(); p = model.predict_proba(b[FEATURES])[:, 1]; pred_s = time.time() - t0
    oof[va] = p
    m = evaluate(b, p, train_pts)
    m.update({"fold": fold, "train_trips": a.trip_id.nunique(), "val_trips": b.trip_id.nunique(),
              "fit_s": fit_s, "predict_s": pred_s})
    cv_rows.append(m)
    print(f"fold {fold}: top-1 {m['top1']:.4f} (nearest-arc {m['top1_nearest_arc']:.4f}) "
          f"| F1 {m['f1']:.4f} | log loss {m['log_loss']:.4f} | fit {fit_s:.1f}s")
    if fold == 4:
        last_fold_model, last_fold_val = model, b   # kept for permutation importance (section 5)

cv = pd.DataFrame(cv_rows)
cols = ["top1", "top1_given_candidate", "top1_nearest_arc", "accuracy", "precision", "recall", "f1", "log_loss", "fit_s", "predict_s"]
cv_summary = cv[cols].agg(["mean", "std"]).T
cv_summary

## 4. Probability calibration (fitted on out-of-fold predictions — train only)
A one-feature logistic regression on the logit of the forest's probability (Platt scaling). Because every out-of-fold prediction comes from a forest that never saw that trip, the calibrator learns how the forest behaves on *unseen* trips — which is what student_test and the evaluator trips are.

In [ ]:
calibrator = LogisticRegression(C=1e6, max_iter=1000).fit(logit(oof), train.label)
oof_cal = calibrator.predict_proba(logit(oof))[:, 1]
print(f"out-of-fold log loss: raw {log_loss(train.label, np.clip(oof, 1e-7, 1 - 1e-7)):.4f} "
      f"-> calibrated {log_loss(train.label, oof_cal):.4f}")

# reliability curve: predicted probability vs observed positive rate, in 10 bins
bins = np.linspace(0, 1, 11)
fig, ax = plt.subplots(figsize=(5.5, 5))
for name, p, col in [("raw forest", oof, "#eb6834"), ("calibrated", oof_cal, "#2a78d6")]:
    b = np.clip(np.digitize(p, bins) - 1, 0, 9)
    d = pd.DataFrame({"b": b, "p": p, "y": train.label.values}).groupby("b").mean()
    ax.plot(d.p, d.y, "o-", color=col, label=name)
ax.plot([0, 1], [0, 1], "--", color="#9c9b96", label="perfect")
ax.set_xlabel("predicted probability"); ax.set_ylabel("observed share of reference arcs")
ax.set_title("Reliability (out-of-fold, train)"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "rf_reliability.png", dpi=150); plt.show()

## 5. Final model — all usable train trips, evaluated once on student_test
`p_test` is the **calibrated** probability (what the HMM will use); the raw forest's log loss is reported alongside.

In [ ]:
final = make_model()
t0 = time.time(); final.fit(train[FEATURES], train.label); final_fit_s = time.time() - t0
t0 = time.time(); p_test_raw = final.predict_proba(test[FEATURES])[:, 1]; final_pred_s = time.time() - t0
p_test = calibrator.predict_proba(logit(p_test_raw))[:, 1]

test_metrics = evaluate(test, p_test, test_pts)
test_metrics.update({"log_loss_uncalibrated": log_loss(test.label, np.clip(p_test_raw, 1e-7, 1 - 1e-7)),
                     "fit_s": final_fit_s, "predict_s": final_pred_s, "rows": len(test)})
pd.Series(test_metrics).round(4)

In [ ]:
# Per-trip top-1 on student_test
per_trip = []
for tid, g in test.assign(p=p_test).groupby("trip_id"):
    per_trip.append({"trip_id": tid, **top1(g, g.p.values, test_pts),
                     "top1_nearest_arc": top1(g, -g.dist_rank.values.astype(float), test_pts)["top1"]})
per_trip = pd.DataFrame(per_trip).sort_values("top1")
per_trip.round(4)

## 6. Feature importance (permutation, on a CROSS-VALIDATION fold — not student_test)
Permutation importance shuffles one feature at a time and measures how much log loss gets worse; it is computed on the held-out trips of CV fold 4, so student_test is never used to judge features. It is used rather than the forest's built-in impurity importance, which favours features with many distinct values, and so that all models are judged the same way.

In [ ]:
sample = last_fold_val.sample(n=min(100_000, len(last_fold_val)), random_state=SEED)
t0 = time.time()
pi = permutation_importance(last_fold_model, sample[FEATURES], sample.label, scoring="neg_log_loss",
                            n_repeats=3, random_state=SEED, n_jobs=-1)
importance = pd.DataFrame({"feature": FEATURES, "mean": pi.importances_mean, "std": pi.importances_std}).sort_values("mean")
print(f"permutation importance in {time.time() - t0:.0f}s (increase in log loss when shuffled)")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(importance.feature, importance["mean"], xerr=importance["std"], color="#2a78d6")
axes[0].set_title("Permutation importance (CV fold 4): log-loss increase")
axes[0].tick_params(axis="y", labelsize=8)
cm = confusion_matrix(test.label, (p_test >= 0.5).astype(int))
ConfusionMatrixDisplay(cm, display_labels=["not ref (0)", "ref arc (1)"]).plot(ax=axes[1], colorbar=False, values_format=",")
axes[1].set_title("Student_test confusion matrix (rows, threshold 0.5)")
fig.tight_layout()
fig.savefig(OUT / "rf_importance_and_confusion.png", dpi=150)
plt.show()

## 7. Comparison with Models 1 and 2 (if their results are present)

In [ ]:
rows = []
for name, folder in [("Logistic Regression", "logistic_regression"),
                     ("HistGradientBoosting", "hist_gradient_boosting"), ("Random Forest", None)]:
    if folder is None:
        cvm, cvs, st = cv[cols].mean(), cv[cols].std(), test_metrics
    else:
        f = RESULTS / folder / "experiment_log.json"
        if not f.exists():
            print(f"(no {f} yet — run that model's notebook to include it)")
            continue
        log = json.loads(f.read_text())
        cvm, cvs, st = pd.Series(log["cv_mean"]), pd.Series(log["cv_std"]), log["student_test"]
    rows.append({"model": name, "cv_top1": f"{cvm['top1']:.4f} ± {cvs['top1']:.4f}",
                 "test_top1": round(st["top1"], 4), "test_f1": round(st["f1"], 4),
                 "test_log_loss": round(st["log_loss"], 4), "cv_fit_s": round(cvm["fit_s"], 1)})
pd.DataFrame(rows)

## 8. Save model, calibrator, experiment log and predictions

In [ ]:
joblib.dump({"forest": final, "calibrator": calibrator}, OUT / "rf_model.joblib", compress=3)

log = {
    "config": CONFIG,
    "features": FEATURES,
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__,
                 "numpy": np.__version__, "pandas": pd.__version__},
    "data": {"train_trips": trips_of("train"), "student_test_trips": trips_of("student_test"),
             "train_rows": len(train), "student_test_rows": len(test)},
    "cv_folds": cv.to_dict(orient="records"),
    "cv_mean": cv[cols].mean().to_dict(),
    "cv_std": cv[cols].std().to_dict(),
    "student_test": test_metrics,
    "student_test_per_trip": per_trip.to_dict(orient="records"),
    "permutation_importance_cv_fold4": importance.to_dict(orient="records"),
    "calibration": {"method": "Platt sigmoid on out-of-fold logit",
                    "coef": float(calibrator.coef_[0, 0]), "intercept": float(calibrator.intercept_[0]),
                    "oof_log_loss_raw": log_loss(train.label, np.clip(oof, 1e-7, 1 - 1e-7)),
                    "oof_log_loss_calibrated": log_loss(train.label, oof_cal)},
}
(OUT / "experiment_log.json").write_text(json.dumps(log, indent=2, default=float))

test[["trip_id", "idx", "arc_id", "label"]].assign(p_rf=p_test, p_rf_uncalibrated=p_test_raw).to_csv(
    OUT / "student_test_predictions.csv.gz", index=False, float_format="%.6f")
print("saved to", OUT)

## 9. (Optional) Map — GPS vs predicted path vs reference route
Needs `Student folder.zip` in `MyDrive/PathSnap/` for coordinates.

In [ ]:
# ---- Map: GPS vs predicted path vs reference route (run after the final-model cell) ----
import folium

TRIP = "00000077"   # any student_test trip: 00000000 07 09 40 52 56 65 76 77 91

# raw coordinates come from the original dataset zip (features hold no coordinates)
RAW_ZIP = Path("/content/drive/MyDrive/PathSnap/Student folder.zip")
RAW = Path("/content/raw") if IN_COLAB else DATA
if IN_COLAB and not (RAW / "Student folder").exists():
    with zipfile.ZipFile(RAW_ZIP) as z:
        z.extractall(RAW)
trip_dir = RAW / "Student folder" / "student_test" / TRIP

track = np.loadtxt(trip_dir / f"{TRIP}.track", ndmin=2)        # lon, lat, t   (row = idx)
nodes = np.loadtxt(trip_dir / f"{TRIP}.nodes", ndmin=2)        # lon, lat      (row = node id)
arcs = np.loadtxt(trip_dir / f"{TRIP}.arcs", dtype=int, ndmin=2)  # from, to   (row = arc id)
route = np.loadtxt(trip_dir / f"{TRIP}.route", dtype=int, ndmin=1)

# predicted arc per observation = highest-probability candidate
g = test[test.trip_id == TRIP].assign(p=p_test[(test.trip_id == TRIP).values])
best = g.loc[g.groupby("idx").p.idxmax(), ["idx", "arc_id", "label"]]
acc = best.label.mean()

def arc_line(a):  # folium wants [lat, lon]
    u, v = arcs[a]
    return [[nodes[u, 1], nodes[u, 0]], [nodes[v, 1], nodes[v, 0]]]

m = folium.Map(location=[track[:, 1].mean(), track[:, 0].mean()], zoom_start=14, tiles="OpenStreetMap")

ref = folium.FeatureGroup("Reference route (green)").add_to(m)
for a in route:
    folium.PolyLine(arc_line(a), color="green", weight=7, opacity=0.45).add_to(ref)

pred = folium.FeatureGroup("Predicted arcs (red)").add_to(m)
for a in best.arc_id.unique():
    folium.PolyLine(arc_line(a), color="red", weight=3, opacity=0.9).add_to(pred)

gps = folium.FeatureGroup("GPS points (blue = correct, orange = wrong)").add_to(m)
for i, ok in zip(best.idx, best.label):
    folium.CircleMarker([track[i, 1], track[i, 0]], radius=2, weight=0, fill=True, fill_opacity=0.8,
                        color="#2a78d6" if ok else "#eb6834", tooltip=f"idx {i}").add_to(gps)

folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[track[:, 1].min(), track[:, 0].min()], [track[:, 1].max(), track[:, 0].max()]])
print(f"Trip {TRIP}: {len(best):,} observations, top-1 {acc:.1%} | wrong points shown in orange")
m.save(str(OUT / f"map_{TRIP}.html"))
m